# 01. Data Quality

This notebook inspects the missing value pattern and the outlier behavior in the equities pretraining universe. It is the artifact that backs the missing value and outlier section of [docs/proposal_revised.md](../docs/proposal_revised.md).

The notebook starts on a synthetic panel so it runs end to end without any external data download. Switch the `LOADER` variable at the top to `KaggleStocksLoader` once the Kaggle dataset is unpacked under `data/raw/Stocks`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

from stockml.data.preprocessing import drop_zero_volume, report_missing, winsorize_returns
from stockml.features.pipeline import build_features
from stockml.utils.io import project_root, read_yaml

rng = np.random.default_rng(0)
dates = pd.bdate_range("2010-01-01", "2024-12-31")
panels = []
for ticker in ["AAA", "BBB", "CCC", "DDD"]:
    rets = rng.normal(0.0005, 0.012, len(dates))
    close = 50.0 * np.exp(np.cumsum(rets))
    df = pd.DataFrame(
        {
            "open": close * (1 + rng.normal(0, 0.001, len(dates))),
            "high": close * (1 + np.abs(rng.normal(0, 0.005, len(dates)))),
            "low": close * (1 - np.abs(rng.normal(0, 0.005, len(dates)))),
            "close": close,
            "volume": rng.integers(1_000_000, 5_000_000, len(dates)),
            "ticker": ticker,
        },
        index=dates,
    )
    df.index.name = "date"
    panels.append(df)
panel = pd.concat(panels)
panel.head()

## Missing values before any cleaning

In [ ]:
report_missing(panel)

## Effect of dropping zero volume rows

In [ ]:
cleaned = drop_zero_volume(panel)
print(f"rows before: {len(panel)}, rows after: {len(cleaned)}")

## Feature pipeline output

In [ ]:
feature_cfg = read_yaml(project_root() / "configs" / "features" / "standard_technicals.yaml")
feats = build_features(cleaned, feature_cfg, market=None)
feats.tail(5)

## Effect of winsorization on the return distribution

In [ ]:
raw = feats["log_return_1"].dropna()
wins = winsorize_returns(feats, return_col="log_return_1")["log_return_1"].dropna()
summary = pd.DataFrame(
    {
        "raw": raw.describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]),
        "winsorized": wins.describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]),
    }
)
summary

## Next steps

Once the Kaggle dataset is unpacked under `data/raw/Stocks` switch the data source at the top of the notebook to `stockml.data.ingestion.KaggleStocksLoader` and rerun. The same code paths apply because the loaders return the same panel schema.